# Building Inventory Generator Notebook

This notebook demonstrates how to:
1. Install BrailsPlusPlus for building footprint data extraction
2. Fetch building inventories for a specific location or bounding box
3. Merge with National Structure Inventory (NSI) data for enriched attributes
4. Visualize the results on interactive maps using Folium
5. Export data to various formats (GeoJSON, CSV)

**Key Features:**
- Automatic data enrichment with NSI attributes (YearBuilt, NumberOfStories, etc.)
- Interactive map visualization
- Multiple export formats for further analysis

## Step 1: Install BrailsPlusPlus

Install the latest version of BrailsPlusPlus from GitHub. This library provides tools for:
- Scraping building footprints from multiple sources (OSM, USA Footprints)
- Integrating with National Structure Inventory (NSI)
- Managing asset inventories

**Note:** This may take a few minutes on first run.

In [ ]:
!pip install --upgrade git+https://github.com/NHERI-SimCenter/BrailsPlusPlus

  Cloning https://github.com/NHERI-SimCenter/BrailsPlusPlus to c:\users\varun\appdata\local\temp\pip-req-build-kxo1uwce
  Resolved https://github.com/NHERI-SimCenter/BrailsPlusPlus to commit 12390ca660e86f13c95faf4bddbbded4f0f01efa
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for brails: filename=brails-4.1.4-py3-none-any.whl size=2317196 sha256=cd2fad26a4e0455776a4974d13314524349b4011fe59d07369816bc8dd2f9866
  Stored in directory: C:\Users\varun\AppData\Local\Temp\pip-ephem-wheel-cache-q6kjw79f\wheels\0d\c3\4e\5d092a6b9f422aabedf42d53704326af544d122b5fcae7fb5e
Successfully built brails
  Attempting uninstall: brails
    Found existing installation: BRAILS 3.1.3
    Uninstalling BRAILS-3.1.

  Running command git clone --filter=blob:none --quiet https://github.com/NHERI-SimCenter/BrailsPlusPlus 'C:\Users\varun\AppData\Local\Temp\pip-req-build-kxo1uwce'


In [ ]:
import numpy as np

## Step 2: Fetch Building Inventory by Location

Run the inventory script with a location name. The script will:
1. Geocode the location name to coordinates
2. Fetch building footprints from OpenStreetMap
3. Merge with NSI data to add attributes like:
   - YearBuilt
   - NumberOfStories
   - StructureType
   - OccupancyClass
4. Save results to `inventory_<location>_nsi.geojson`

**Alternative:** You can also use a bounding box (see next cell) for more precise area control.

In [ ]:
!python get_inventory_simple.py --location "Reseda, CA"


Searching for Reseda, CA...
Found Reseda, Los Angeles, Los Angeles County, California, United States

Searching for Reseda, CA...
Found Reseda, Los Angeles, Los Angeles County, California, United States

Found a total of 20670 building footprints in Reseda
Footprints retrieved: 20670
Merging NSI in chunks of ~1500 (total 20670 assets) ...

Getting National Structure Inventory (NSI) building data for the entered location...
Found a total of 867 building points in NSI that match the footprint data.
  NSI merged: 1500/20670

Getting National Structure Inventory (NSI) building data for the entered location...
Found a total of 1024 building points in NSI that match the footprint data.
  NSI merged: 3000/20670

Getting National Structure Inventory (NSI) building data for the entered location...
Found a total of 698 building points in NSI that match the footprint data.
  NSI merged: 4500/20670

Getting National Structure Inventory (NSI) building data for the entered location...
Found a total

## Alternative: Fetch by Bounding Box

Use this approach when you need precise geographic boundaries.

**Bounding Box Format:** `--bbox LON_MIN LAT_MIN LON_MAX LAT_MAX`
- All coordinates in decimal degrees
- Example: `-118.60 34.20 -118.45 34.30` covers part of Northridge, CA

**Advantages:**
- More precise area control
- Useful for rectangular study areas
- Better for comparing specific regions

Uncomment the cell below to use bounding box mode instead of location name.

In [ ]:
!python get_inventory_simple.py --bbox -118.60 34.20 -118.58 34.32
#--bbox LON_MIN LAT_MIN LON_MAX LAT_MAX

^C


## Step 3: Visualize Buildings on Interactive Map

Create an interactive Folium map to visualize the building footprints.

**What this cell does:**
1. Loads the GeoJSON file containing building inventory
2. Calculates the map center from building centroids
3. Creates a Folium map with buildings overlaid
4. Saves the interactive map as HTML for viewing in a browser

**Output:**
- Interactive map displayed in notebook
- HTML file saved to `map/reseda_loc_nsi_map.html`

**Note:** You can open the HTML file in any browser to interact with the map (zoom, pan, click features).

In [ ]:
# Import required libraries for mapping and geospatial analysis
import folium
import geopandas as gpd
from shapely.geometry import mapping
import matplotlib.pyplot as plt
import os

# Load the GeoJSON file containing building inventory with NSI attributes
gdf = gpd.read_file("data/reseda_loc_nsi.geojson")

# Calculate the center point of the map using the mean of all building centroids
# This ensures the map is centered on the study area
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]

# Create a Folium map centered on the study area
# zoom_start=14 provides a good street-level view
m = folium.Map(location=center, zoom_start=14)

# Add the building footprints to the map as a GeoJSON layer
# This will display all buildings with their geometries
folium.GeoJson(gdf).add_to(m)

# Display the map in the notebook
m

# Create the map directory if it doesn't exist
os.makedirs("map", exist_ok=True)

# Save the interactive map as an HTML file
# You can open this file in any web browser for full interactivity
m.save("map/reseda_loc_nsi_map.html")

C:\Users\varun\AppData\Local\Temp\ipykernel_14120\450082874.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]


## Step 4: Export to CSV Format

Convert the GeoDataFrame to CSV for analysis in spreadsheet software or other tools.

**What gets exported:**
- All building attributes (NSI data)
- Geometry as WKT (Well-Known Text) format
- All numeric and text fields

**Use cases:**
- Statistical analysis in Excel, R, or Python pandas
- Database import
- Sharing data with non-GIS users

**Output:** `csv_data/reseda_loc_nsi.csv`

In [ ]:
# Create the output directory for CSV files if it doesn't exist
os.makedirs("csv_data", exist_ok=True)

# Convert the GeoDataFrame to CSV format
# index=False prevents adding an extra index column
# Geometry will be exported as WKT (Well-Known Text) format
gdf.to_csv("csv_data/reseda_loc_nsi.csv", index=False)

print(f"✅ Exported {len(gdf)} buildings to CSV")
print(f"📁 File saved to: csv_data/reseda_loc_nsi.csv")

## Additional Analysis (Optional)

Use this section to perform additional analysis on the building inventory:

**Suggested analyses:**
- Summary statistics of building attributes
- Year built distribution histograms
- Number of stories analysis
- Occupancy type breakdown
- Spatial clustering analysis

**Example code snippets:**

```python
# View first few rows
print(gdf.head())

# Summary statistics
print(gdf.describe())

# Count by year built decade
if 'YearBuilt' in gdf.columns:
    gdf['Decade'] = (gdf['YearBuilt'] // 10) * 10
    print(gdf['Decade'].value_counts().sort_index())

# Plot building age distribution
if 'YearBuilt' in gdf.columns:
    gdf['YearBuilt'].hist(bins=30)
    plt.xlabel('Year Built')
    plt.ylabel('Count')
    plt.title('Building Age Distribution')
    plt.show()
```

In [ ]:
# If you don't have ipyleaflet/ipywidgets installed, uncomment and run the next line:
# !pip install ipyleaflet ipywidgets

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 18.0 MB/s eta 0:00:00


## GPS Points Generator for Bounding Boxes

In [ ]:
from ipyleaflet import Map, Marker
import pandas as pd
from ipywidgets import Button, VBox, HTML
from IPython.display import display

# If you don't have ipyleaflet/ipywidgets installed, uncomment and run the next line:
# !pip install ipyleaflet ipywidgets

# reuse existing center variable (lat, lon)
map_center = center

# create an interactive ipyleaflet map (do not overwrite existing folium 'm')
interactive_map = Map(center=map_center, zoom=14, scroll_wheel_zoom=True, layout={'height': '600px'})

# storage for clicked points and a DataFrame view
_clicked_points = []
clicked_points_df = pd.DataFrame(columns=['lat', 'lon'])

status = HTML(value="Click on the map to add points. Points: 0")

def _on_map_interaction(**event):
    if event.get('type') == 'click':
        lat, lon = event.get('coordinates')  # ipyleaflet gives (lat, lon)
        _clicked_points.append({'lat': lat, 'lon': lon})
        # add a marker on the map for feedback
        marker = Marker(location=(lat, lon))
        interactive_map.add_layer(marker)
        # update DataFrame and status
        global clicked_points_df
        clicked_points_df = pd.DataFrame(_clicked_points)
        status.value = f"Points: {len(_clicked_points)}"

interactive_map.on_interaction(_on_map_interaction)

# Buttons to save / clear points
save_btn = Button(description='Save to CSV')
clear_btn = Button(description='Clear points')

def _save_points(b):
    if not _clicked_points:
        status.value = "No points to save."
        return
    clicked_points_df.to_csv("clicked_points.csv", index=False)
    status.value = f"Saved {len(_clicked_points)} points to clicked_points.csv"

def _clear_points(b):
    _clicked_points.clear()
    # remove markers (Marker layers) from map
    for layer in list(interactive_map.layers):
        if isinstance(layer, Marker):
            interactive_map.remove_layer(layer)
    global clicked_points_df
    clicked_points_df = pd.DataFrame(columns=['lat', 'lon'])
    status.value = "Cleared points"

save_btn.on_click(_save_points)
clear_btn.on_click(_clear_points)

# Display the map, status and controls. The DataFrame `clicked_points_df` is available in the notebook.
display(VBox([interactive_map, status, save_btn, clear_btn]))


In [ ]:
#For Northridge Building Inventory
# !python get_inventory_simple.py --bbox 34.2203 -118.5622 34.2575 -118.502152

In [ ]:
#find the min and max lat and lon in lat,lon pairs below the points below are for our entire study area
import numpy as np
lat_long_pairs = np.array([[33.48597686345578,-117.56517817612001],
[34.72716587525696,-117.56000958738024],
[34.761470829477474,-119.3995565766095],
[33.71428689710177,-118.43873971294566],
[35.00930211335558,-118.1662759400126]])
min_lat = np.min(lat_long_pairs[:,0])
min_lon = np.min(lat_long_pairs[:,1])
max_lat = np.max(lat_long_pairs[:,0])
max_lon = np.max(lat_long_pairs[:,1])

print("Min Latitude:", min_lat)
print("Min Longitude:", min_lon)
print("Max Latitude:", max_lat)
print("Max Longitude:", max_lon)

Min Latitude: 33.48597686345578
Min Longitude: -119.3995565766095
Max Latitude: 35.00930211335558
Max Longitude: -117.56000958738024


In [ ]:
lat_steps = np.arange(min_lat, max_lat, 0.1)
lon_steps = np.arange(min_lon, max_lon, 0.1)
count = 0
for i in range(len(lat_steps)-1):
    for j in range(len(lon_steps)-1):
        lat_min = lat_steps[i]
        lat_max = lat_steps[i+1]
        lon_min = lon_steps[j]
        lon_max = lon_steps[j+1]
        print(f"Processing bbox: {lat_min}, {lon_min}, {lat_max}, {lon_max}")
        # !python get_inventory_simple.py --bbox {lon_min} {lat_min} {lon_max} {lat_max}
        count += 1
print(f"Total bounding boxes processed: {count}")

Processing bbox: 33.48597686345578, -119.3995565766095, 33.585976863455784, -119.2995565766095
Processing bbox: 33.48597686345578, -119.2995565766095, 33.585976863455784, -119.19955657660951
Processing bbox: 33.48597686345578, -119.19955657660951, 33.585976863455784, -119.09955657660952
Processing bbox: 33.48597686345578, -119.09955657660952, 33.585976863455784, -118.99955657660952
Processing bbox: 33.48597686345578, -118.99955657660952, 33.585976863455784, -118.89955657660953
Processing bbox: 33.48597686345578, -118.89955657660953, 33.585976863455784, -118.79955657660953
Processing bbox: 33.48597686345578, -118.79955657660953, 33.585976863455784, -118.69955657660954
Processing bbox: 33.48597686345578, -118.69955657660954, 33.585976863455784, -118.59955657660954
Processing bbox: 33.48597686345578, -118.59955657660954, 33.585976863455784, -118.49955657660955
Processing bbox: 33.48597686345578, -118.49955657660955, 33.585976863455784, -118.39955657660956
Processing bbox: 33.4859768634557

In [ ]:
# Split into 4 equal parts by dividing latitude range
total_lat_steps = len(lat_steps) - 1
quarter_size = total_lat_steps // 4

# Choose which quarter to run (0, 1, 2, or 3)
quarter = 0  # Change this to 0, 1, 2, or 3 for each person

# Calculate start and end indices for this quarter
start_i = quarter * quarter_size
if quarter == 3:  # Last quarter takes any remainder
    end_i = total_lat_steps
else:
    end_i = (quarter + 1) * quarter_size

print(f"Running quarter {quarter + 1}/4")
print(f"Latitude indices: {start_i} to {end_i}")

count = 0
for i in range(start_i, end_i):
    for j in range(len(lon_steps)-1):
        lat_min = lat_steps[i]
        lat_max = lat_steps[i+1]
        lon_min = lon_steps[j]
        lon_max = lon_steps[j+1]
        print(f"Processing bbox: {lat_min}, {lon_min}, {lat_max}, {lon_max}")
        !python get_inventory_simple.py --bbox {lon_min} {lat_min} {lon_max} {lat_max}
        count += 1
print(f"Quarter {quarter + 1} complete. Processed {count} bounding boxes.")